# Mini-project: compare Rongowai with L-band NISAR data and soil moisture estimates

The premise of this project is simple: compare data from each data source for the same locations them and provide interpration. 

Spatially and temporally co-located data already downloaded:

| Date/ location                 | Rongowai | NISAR    | Notes |
| ------------------------------ | -------- | -------- | ----- |
| 20251224: Christchurch region  | 2 Rongowai flights: 20251224-161331_NZCH-NZNP_L1.nc, 20251224-180710_NZNP-NZCH_L1.nc (Christchurch to New Plymouth, return)     | NISAR_L2_PR_GCOV_008_123_A_156_2005_DHDH_A_20251224T183611_20251224T183646_X05009_N_F_J_001.h5, NISAR_L3_PR_SME2_008_123_A_156_2005_DHDH_A_20251224T183611_20251224T183646_X05010_N_F_J_001.h5     | 1st flight ~2 hours ahead of image, 2nd flight ~during or just after image capture |
| 20260110: Auckland region      | 1 flight: 20260110-065500_NZGS-NZAA_L1.nc (Gisborne to Auckland)     | NISAR_L2_PR_GCOV_010_015_D_107_2005_DHDH_A_20260110T062028_20260110T062106_X05009_N_F_J_001.h5, NISAR_L3_PR_SME2_010_015_D_107_2005_DHDH_A_20260110T062028_20260110T062106_X05010_N_F_J_001.h5     | Flight ~1 hour after image capture |
| 20260115: Wellington-Tauranga  | 20260115-171000_NZWN-NZTG_L1.nc, 20260115-190324_NZTG-NZWN_L1.nc (Wellington to Tauranga, return)     |  NISAR_L2_PR_GCOV_010_094_A_157_2005_DHDH_A_20260115T182016_20260115T182043_X05010_N_F_J_001.h5, NISAR_L2_PR_GCOV_010_094_A_158_2005_DHDH_A_20260115T182042_20260115T182117_X05010_N_F_J_001.h5, NISAR_L3_PR_SME2_010_094_A_157_2005_DHDH_A_20260115T182016_20260115T182043_X05010_N_F_J_001.h5, NISAR_L3_PR_SME2_010_094_A_158_2005_DHDH_A_20260115T182042_20260115T182117_X05010_N_F_J_001.h5      | 1st flight ~1 hour before image capture; 2nd flight ~1 hour after image capture

Note:
- NISAR_L2_PR_GCOV refers to Level 2 Geocoded Polarimetric Covariance (GCOV), https://nisar-docs.asf.alaska.edu/gcov/
- NISAR_L3_PR_SME2 refers to Level 3 soil moisture estimates, https://nisar-docs.asf.alaska.edu/sme2/

The NISAR data were downloaded from: https://www.earthdata.nasa.gov/data/catalog/asf-nisar-l2-gcov-beta-v1-1 and https://www.earthdata.nasa.gov/data/catalog/asf-nisar-l3-sme2-beta-v1-1.

## Getting started

You'll need to develop some code to do the interpretation yourself. Below is enough to get you started:
- Load and visualise L2 NISAR data as RGB
- Extract NISAR values at Rongowai points

Note that you can load the NISAR data directly in QGIS (drag and drop onto map window), but first you should change the file suffix from ```.h5``` to ```.nc```. See here for more details and advice: https://nisar-docs.asf.alaska.edu/using-overview/ 

Load packages and set up:

In [ ]:
import rongowai_helpers as rongowai # required for importing data and the plotting functions used
#import load_gcov as load_gcov

import pandas as pd
import geopandas as gpd
import xarray as xr
import netCDF4 as nc
import numpy as np

from lonboard import Map, BitmapLayer, RasterLayer

import re
from IPython.display import display, clear_output, HTML
from itables import show
import itables.options as opt
import ipywidgets as widgets

from pathlib import Path
import pydeck as pdk
from tqdm import tqdm
import random

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import os

from osgeo import gdal
import rasterio
import rioxarray

verbose = True

nisar_data_root = Path("../NISAR")
rongowai_data_root = Path("./data/L1")

Set file names: you'll need to edit this template to load all the data in:

In [ ]:
# Just picking three files here to test the functions on. The NISAR files are from the same granule, and the Rongowai file is from the same day.
nisar_gcov_file = nisar_data_root / "NISAR_L2_PR_GCOV_008_123_A_156_2005_DHDH_A_20251224T183611_20251224T183646_X05009_N_F_J_001.h5"
nisar_sme_file = nisar_data_root / "NISAR_L3_PR_SME2_008_123_A_156_2005_DHDH_A_20251224T183611_20251224T183646_X05010_N_F_J_001.h5"
rongowai_file = rongowai_data_root / "20251224-180710_NZNP-NZCH_L1.nc"

For the selected Rongowai flight, extract data from netcdf and produce gpkg files, load:

In [3]:
gpkg_file = Path(rongowai_file).parent / f"{os.path.basename(rongowai_file)}.gpkg"
_ = rongowai.l1_to_gpkg_single(rongowai_file, gpkg_file)
gdf = gpd.read_file(gpkg_file, layer = 'spatial') # 2D point data for each GNSS-R return
gdf_ac = gpd.read_file(gpkg_file, layer = 'ac') # point data for the aircraft's position at each timestamp
gdf_flightvector = gpd.read_file(gpkg_file, layer = 'flightvector') # a polyline of the flight path

Open and list variables in a NISAR netcdf:

In [4]:
ds = xr.open_dataset(nisar_gcov_file, group="science/LSAR/GCOV/grids/frequencyA")
print(ds)

<xarray.Dataset> Size: 5GB
Dimensions:                (yCoordinates: 15840, xCoordinates: 16416,
                            phony_dim_27: 2)
Coordinates:
  * yCoordinates           (yCoordinates) float64 127kB 5.389e+06 ... 5.072e+06
  * xCoordinates           (xCoordinates) float64 131kB 4.154e+05 ... 7.438e+05
Dimensions without coordinates: phony_dim_27
Data variables:
    HHHH                   (yCoordinates, xCoordinates) float32 1GB ...
    HVHV                   (yCoordinates, xCoordinates) float32 1GB ...
    listOfCovarianceTerms  (phony_dim_27) <U4 32B ...
    listOfPolarizations    (phony_dim_27) <U2 16B ...
    mask                   (yCoordinates, xCoordinates) float32 1GB ...
    numberOfLooks          (yCoordinates, xCoordinates) float32 1GB ...
    numberOfSubSwaths      uint8 1B ...
    projection             uint32 4B ...
    rtcGammaToSigmaFactor  (yCoordinates, xCoordinates) float32 1GB ...
    xCoordinateSpacing     float64 8B ...
    yCoordinateSpacing     float6

Check the projection information an save an EPSG string for use below: 

In [ ]:
#read the projection information for the frequencyA grid
projection = ds["projection"][()]
projection

<xarray.DataArray 'projection' ()> Size: 4B
[1 values with dtype=uint32]
Attributes: (12/14)
    epsg_code:                         32759
    spatial_ref:                       PROJCS["WGS 84 / UTM zone 59S",GEOGCS[...
    grid_mapping_name:                 transverse_mercator
    semi_major_axis:                   6378137.0
    inverse_flattening:                298.257223563
    ellipsoid:                         WGS84
    ...                                ...
    longitude_of_projection_origin:    0.0
    latitude_of_projection_origin:     0.0
    utm_zone_number:                   59
    longitude_of_central_meridian:     171.0
    scale_factor_at_central_meridian:  0.9996
    description:                       Product map grid projection: EPSG code...

In [8]:
crs = f"EPSG:{int(projection.epsg_code)}"
crs

'EPSG:32759'

## Extract netcdf data and do some band math

Extract the primary data bands, ensure the CRS is included, and calculate a band ratio (for RGB visualisation): 

In [ ]:
hh = ds['HHHH']
hv = ds['HVHV']

# set CRS
hh.rio.write_crs(crs, inplace=True)
hv.rio.write_crs(crs, inplace=True)

# calc band ratio
ratio = hh / (hv + 1e-6) # add a small number to the denominator to avoid division by zero, which can happen in areas with no signal. This is a common practice when calculating ratios of radar backscatter coefficients, to prevent undefined values. The choice of 1e-6 is somewhat arbitrary, but it should be small enough to not significantly affect the ratio in areas with valid signal, while still preventing division by zero in areas with no signal.
if ratio.rio.crs is None:
    ratio.rio.write_crs(hh.rio.crs, inplace=True)

ratio.attrs['Description'] = "Polarimetric ratio (HH/HV)"
ratio.attrs['Formula'] = "hh / (hv + 1e-6)"
ratio.name = "HH-HV-Ratio"

## Export 3-band NISAR tiff image for RGB visualisation

Because of the size of the image data, it is not easy to visualise them in a Jupyter notebook. It is recommended instead to load the netcdf data within QGIS (change the file suffix from ```.h5``` to ```.nc``` for the spatial geometry to be correctly interpreted). To aid visualisation, you can create a false colour composite and open the resulting geotiff file in QGIS:

In [ ]:
# Create stack:
rgb_stack = xr.concat([hh, hv, ratio], dim=xr.Variable("band", ["HH", "HV", "HH-HV-Ratio"]))

# Loop through and merge attributes from all sources - note this won't preserve everything, but these are still in the individual DataArrays if needed
for da in [hh, hv, ratio]:
    rgb_stack.attrs.update(da.attrs)

gcov_rgb_output_file =  data_root / f"{nisar_gcov_file}_rgb.tif"
rgb_stack.rio.to_raster(gcov_rgb_output_file)

Run the code below to ensure the bands are labelled correctly in QGIS: 

In [52]:
import rasterio
with rasterio.open(gcov_rgb_output_file, "r+") as dst:
    # This must be a tuple with exactly one name per band
    dst.descriptions = ("HH", "HV", "HH/HV Ratio")

## Extract values at Rongowai point locations

First, select good Rongowai data through filtering, including selecting only one poliarisation (```ddm_ant==2```): 

In [ ]:
# define some reasonable filters to select quality Rongowai observations
filter_dict = {
        "coherence_metric":    (1, 9999), # To select coherent returns, set the lower bound to 1 (since coherence_metric is 0 for incoherent returns and >1 for coherent returns). 
        "sp_inc_angle":        (0, 60),  # Specular incidence angle under 60 degrees should be good quality, but you can adjust this threshold as needed
        "ddm_snr":             (-10.0, 9999), # DDM SNR over -10 dB should be good quality, but you can adjust this threshold as needed
        "ddm_ant":             (2, 2),  # LHCP only
        "sp_surface_type":     (0, 7),  # -1 = ocean, 1 = artifical, 2 = barely vegetated, 3 = inland water, 4 = crop, 5 = grass, 6 = shrub, 7 = forest
    }

# apply the filter to the data (not that plot_rongowai_interactive can do this filtering internally as well, but we'll do it here as we'll us the data again below)
filter_mask = pd.Series(True, index=gdf.index)
for col, (lo, hi) in filter_dict.items():
    if col in gdf.columns:
        filter_mask &= gdf[col].between(lo, hi) | gdf[col].isna()
filtered_gdf = gdf[filter_mask].copy()

# Spatially sort the points to minimize jumps in the raster sampling, which can improve performance
filtered_gdf = filtered_gdf.sort_values("geometry").reset_index(drop=True)

This reduces the amount of data we have to work with, and ensures we're only comparing the NISAR data to high-quality Rongowai observations:

In [16]:
print(f"gdf length: {len(gdf)}, filtered_gdf length: {len(filtered_gdf)}")

gdf length: 88668, filtered_gdf length: 6025


Reproject the points to the same CRS as the raster NISAR data (not the other way around), spatially sort the data, and extract coordinate indexers:

In [ ]:
# Ensure the CRS of your points matches the raster data
# This is critical for spatial alignment. Reprojecting the points is faster and more accurate than reprojecting the raster, especially as the raster is large.
points_gdf = filtered_gdf.to_crs(rgb_stack.rio.crs)

# Create indexers from the point coordinates
x_coords = xr.DataArray(points_gdf.geometry.x, dims="point")
y_coords = xr.DataArray(points_gdf.geometry.y, dims="point")

Do the sampling and save to a DataFrame:

In [57]:
sampledpixels = rgb_stack.sel(xCoordinates=x_coords, yCoordinates=y_coords, method="nearest")
results_df = sampledpixels.to_dataset(dim='band').to_dataframe().reset_index(drop=True)
results_df.columns = [f"NISAR_{name}" for name in results_df.columns] # Prefix the column names ready for a join below
results_df.head()

,NISAR_HH,NISAR_HV,NISAR_HH-HV-Ratio,NISAR_xCoordinates,NISAR_yCoordinates,NISAR_projection,NISAR_spatial_ref
0,0.243559,0.015222,15.999325,632750.0,5184930.0,0,0
1,0.077475,0.035501,2.182260,632850.0,5185050.0,0,0
2,0.043055,0.014943,2.881069,632810.0,5185870.0,0,0
3,0.029204,0.008499,3.435934,632730.0,5185730.0,0,0
4,0.152447,0.015323,9.948130,632870.0,5185950.0,0,0


Join back to original filtered data:

In [ ]:
filtered_gdf = filtered_gdf.join(results_df)
filtered_gdf.head()

Now you can explore the data using the visualisation code in the Rongowai Notebook. Don't forget to extract the soil moisture estimations in the same way (Level 3 NISAR data). Be sure to check out multiple images, look at different factors affecting each signal, such as different land cover types.